In [ ]:
import torch
import matplotlib.pyplot as plt

from metrics import (
    align_channels,
    abs_error_map,
    mse_map,
    darcy_residual_map,
    darcy_match_map,
)


CHANNEL_NAMES = {
    0: "K",
    1: "P",
    2: "phi",
}


def unpack_dataset_item(item):
    """
    Handles both:
        Full dataset: feat, label
        Limited dataset: feat, label, mask
    """
    if len(item) == 3:
        feat, label, mask = item
    else:
        feat, label = item
        mask = None

    return feat, label, mask


def get_model_prediction(model, feat, device):
    model.eval()

    with torch.no_grad():
        feat_b = feat.unsqueeze(0).to(device)
        pred = model(feat_b).cpu().squeeze(0)

    return pred


def compute_heatmap_for_item(
    model,
    dataset,
    idx,
    device,
    metric="abs_error",
    channel=0,
):
    """
    Produces one heatmap for one dataset item.

    metric options:
        "abs_error"
        "squared_error"
        "darcy_pred"
        "darcy_true"
        "darcy_match"
    """

    item = dataset[idx]
    feat, label, mask = unpack_dataset_item(item)

    pred = get_model_prediction(model, feat, device)

    # Add batch dimension for metric functions
    pred_b = pred.unsqueeze(0)
    label_b = label.unsqueeze(0)

    label_b = align_channels(label_b, pred_b)

    if metric == "abs_error":
        heat = abs_error_map(pred_b, label_b)[0, channel]

    elif metric == "squared_error":
        heat = mse_map(pred_b, label_b)[0, channel]

    elif metric == "darcy_pred":
        heat = torch.abs(darcy_residual_map(pred_b))[0, 0]

    elif metric == "darcy_true":
        heat = torch.abs(darcy_residual_map(label_b))[0, 0]

    elif metric == "darcy_match":
        heat = darcy_match_map(pred_b, label_b)[0, 0]

    else:
        raise ValueError(f"Unknown metric: {metric}")

    return heat


def average_heatmap(
    model,
    dataset,
    indices,
    device,
    metric="abs_error",
    channel=0,
):
    """
    Averages heatmaps across multiple dataset indices.
    """

    heatmaps = []

    for idx in indices:
        heat = compute_heatmap_for_item(
            model=model,
            dataset=dataset,
            idx=idx,
            device=device,
            metric=metric,
            channel=channel,
        )
        heatmaps.append(heat)

    return torch.stack(heatmaps).mean(dim=0)


def plot_heatmap(
    heatmap,
    title="Heatmap",
    save_path=None,
):
    plt.figure(figsize=(6, 5))
    plt.imshow(heatmap.numpy())
    plt.colorbar()
    plt.title(title)
    plt.axis("off")

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight", dpi=200)

    plt.show()


def plot_single_model_heatmap(
    model,
    dataset,
    indices,
    device,
    metric="abs_error",
    channel=0,
    save_path=None,
):
    """
    Main function you probably want to call.

    Example:
        plot_single_model_heatmap(
            model,
            val_data,
            indices=range(0, 50),
            device=DEVICE,
            metric="abs_error",
            channel=0
        )
    """

    heatmap = average_heatmap(
        model=model,
        dataset=dataset,
        indices=indices,
        device=device,
        metric=metric,
        channel=channel,
    )

    if metric in ["abs_error", "squared_error"]:
        channel_name = CHANNEL_NAMES.get(channel, f"channel {channel}")
        title = f"Average {metric} heatmap for {channel_name}"
    else:
        title = f"Average {metric} heatmap"

    plot_heatmap(
        heatmap,
        title=title,
        save_path=save_path,
    )

    return heatmap

In [ ]:
heatmap = plot_single_model_heatmap(
    model=model,
    dataset=val_loader.dataset,
    indices=range(0, 50),
    device=DEVICE,
    metric="abs_error",
    channel=0,   # 0 = K, 1 = P
    save_path="heatmap_abs_error_K.png"
)